### Process raw DoReCo

In [4]:
#!/usr/bin/env python3

"""
Process DoReCo corpora into a multilingual dataset.

Input structure (Downloads/):
    stan1290.zip
    doreco_stan1290_core_v2.zip

After extraction:
    stan1290/
        <audio_subdir>/
            *.wav

    doreco_stan1290_core_v2.0/
        *.TextGrid

Output:
    processed/
        stan1290/
            audio/
                *.wav
            alignments/
                *.TextGrid
            chunks.csv

The processed TextGrids retain only:
    tx, wd, ph

Special DoReCo labels enclosed in angle brackets are replaced by "".

chunks.csv columns:
    filename,start,end,text
"""

import csv
import re
import shutil
import zipfile
from pathlib import Path

import textgrid


DOWNLOADS = Path("downloads")
OUTPUT = Path("processed")

KEEP_TIERS = {"tx", "wd", "ph"}

# Matches:
#   <p:>
#   <<fp>>
#   <<ui>word>
#   <<fm>english>
# etc.
SPECIAL_LABEL_RE = re.compile(r"<.*?>")

# ---------------------------------------------------------------------
# Helpers
# ---------------------------------------------------------------------


def clean_label(text):
    """
    Replace DoReCo special labels with empty string.

    Examples:
        <<fp>>            -> ""
        <<ui>word>        -> ""
        <p:>              -> ""
        hello             -> hello
    """
    if text is None:
        return ""

    text = text.strip()

    if "<" in text and ">" in text:
        return ""

    return text


def extract_zip(zip_path, target_dir):
    """
    Extract only if target directory does not already exist.
    """
    if target_dir.exists():
        return

    print(f"Extracting {zip_path.name}")

    with zipfile.ZipFile(zip_path, "r") as zf:
        zf.extractall(target_dir)


def find_single_subdir(path):
    """
    Audio zip contains one subdirectory.
    Return that subdirectory.
    """
    subs = [p for p in path.iterdir() if p.is_dir()]

    if len(subs) == 1:
        return subs[0]

    return path


def find_tier(tg, tier_name):
    """
    Find tier whose name starts with tier_name
    (e.g. tx, wd, ph).
    """
    for tier in tg.tiers:
        name = tier.name.strip()

        if name == tier_name:
            return tier

        if name.startswith(tier_name + "@"):
            return tier

    return None


# ---------------------------------------------------------------------
# Main processing
# ---------------------------------------------------------------------


def process_language(lang_zip):
    """
    lang_zip example:
        stan1290.zip
    """

    lang = lang_zip.stem

    print(f"\nProcessing {lang}")

    core_zip_candidates = list(
        DOWNLOADS.glob(f"doreco_{lang}_core*.zip")
    )

    if not core_zip_candidates:
        print(f"  No core zip found for {lang}")
        return

    core_zip = core_zip_candidates[0]

    audio_extract = DOWNLOADS / lang
    core_extract = DOWNLOADS / core_zip.stem

    extract_zip(lang_zip, audio_extract)
    extract_zip(core_zip, core_extract)

    audio_root = find_single_subdir(audio_extract)

    wavs = list(audio_root.rglob("*.wav"))
    textgrids = list(core_extract.rglob("*.TextGrid"))

    tg_lookup = {p.stem: p for p in textgrids}

    out_lang = OUTPUT / lang
    out_audio = out_lang / "audio"
    out_align = out_lang / "alignments"

    out_audio.mkdir(parents=True, exist_ok=True)
    out_align.mkdir(parents=True, exist_ok=True)

    chunk_rows = []

    for wav in wavs:

        tg_path = tg_lookup.get(wav.stem)

        if tg_path is None:
            print(f"  Missing TextGrid: {wav.stem}")
            continue

        shutil.copy2(wav, out_audio / wav.name)

        tg = textgrid.TextGrid()
        tg.read(str(tg_path))

        new_tg = textgrid.TextGrid(maxTime=tg.maxTime)

        # ---------------------------------------------------------
        # tx tier
        # ---------------------------------------------------------
        tx_tier = find_tier(tg, "tx")

        if tx_tier:
            new_tx = textgrid.IntervalTier(
                name="tx",
                minTime=tx_tier.minTime,
                maxTime=tx_tier.maxTime,
            )

            for iv in tx_tier.intervals:
                cleaned = clean_label(iv.mark)

                new_tx.add(
                    iv.minTime,
                    iv.maxTime,
                    cleaned,
                )

                if cleaned.strip():
                    chunk_rows.append(
                        {
                            "filename": wav.name,
                            "start": iv.minTime,
                            "end": iv.maxTime,
                            "text": cleaned,
                        }
                    )

            new_tg.append(new_tx)

        # ---------------------------------------------------------
        # wd tier
        # ---------------------------------------------------------
        wd_tier = find_tier(tg, "wd")

        if wd_tier:
            new_wd = textgrid.IntervalTier(
                name="wd",
                minTime=wd_tier.minTime,
                maxTime=wd_tier.maxTime,
            )

            for iv in wd_tier.intervals:
                new_wd.add(
                    iv.minTime,
                    iv.maxTime,
                    clean_label(iv.mark),
                )

            new_tg.append(new_wd)

        # ---------------------------------------------------------
        # ph tier
        # ---------------------------------------------------------
        ph_tier = find_tier(tg, "ph")

        if ph_tier:
            new_ph = textgrid.IntervalTier(
                name="ph",
                minTime=ph_tier.minTime,
                maxTime=ph_tier.maxTime,
            )

            for iv in ph_tier.intervals:
                new_ph.add(
                    iv.minTime,
                    iv.maxTime,
                    clean_label(iv.mark),
                )

            new_tg.append(new_ph)

        new_tg.write(str(out_align / tg_path.name))

    # -------------------------------------------------------------
    # chunks.csv
    # -------------------------------------------------------------
    csv_path = out_lang / "chunks.csv"

    with open(csv_path, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(
            f,
            fieldnames=[
                "filename",
                "start",
                "end",
                "text",
            ],
        )

        writer.writeheader()

        for row in chunk_rows:
            writer.writerow(row)

    print(
        f"  {len(wavs)} wavs, "
        f"{len(chunk_rows)} sentence chunks"
    )


# ---------------------------------------------------------------------
# Entry
# ---------------------------------------------------------------------


def main():
    OUTPUT.mkdir(exist_ok=True)

    language_zips = []

    for z in DOWNLOADS.glob("*.zip"):
        if z.name.startswith("doreco_"):
            continue
        language_zips.append(z)

    language_zips = sorted(language_zips)

    for lang_zip in language_zips:
        process_language(lang_zip)

    print("\nDone.")


if __name__ == "__main__":
    main()


Processing anal1239
Extracting anal1239.zip
Extracting doreco_anal1239_core_v2.zip
  23 wavs, 2856 sentence chunks

Processing apah1238
Extracting apah1238.zip
Extracting doreco_apah1238_core_v2.zip
  9 wavs, 2165 sentence chunks

Processing arap1274
Extracting arap1274.zip
Extracting doreco_arap1274_core_v2.zip
  11 wavs, 1754 sentence chunks

Processing bain1259
Extracting bain1259.zip
Extracting doreco_bain1259_core_v2.zip
  17 wavs, 2472 sentence chunks

Processing beja1238
Extracting beja1238.zip
Extracting doreco_beja1238_core_v2.zip
  58 wavs, 6516 sentence chunks

Processing bora1263
Extracting bora1263.zip
Extracting doreco_bora1263_core_v2.zip
  9 wavs, 1224 sentence chunks

Processing cabe1245
Extracting cabe1245.zip
Extracting doreco_cabe1245_core_v2.zip
  39 wavs, 1230 sentence chunks

Processing cash1254
Extracting cash1254.zip
Extracting doreco_cash1254_core_v2.zip
  7 wavs, 2044 sentence chunks

Processing dolg1241
Extracting dolg1241.zip
Extracting doreco_dolg1241_cor

### Create DoReCo Dataset

In [42]:
import re
import numpy as np
import pandas as pd
import soundfile as sf
import librosa
from pathlib import Path
from textgrid import TextGrid


TARGET_SR = 16000


def normalize_text(text):
    text = str(text)

    text = text.replace(".", " ")
    text = text.replace(",", " ")
    text = text.replace("/", " ")

    text = re.sub(r"\s+", " ", text)
    text = text.strip()

    return text


def valid_chunk(text):
    if not text:
        return False

    if text == "****":
        return False

    if len(text.split()) <= 1:
        return False

    return True


def merge_empty_intervals(intervals):
    """
    intervals:
        [(label,start,end), ...]

    merges adjacent empty intervals
    """

    if not intervals:
        return intervals

    merged = [intervals[0]]

    for lab, s, e in intervals[1:]:

        plab, ps, pe = merged[-1]

        if lab == "" and plab == "" and abs(pe - s) < 1e-6:
            merged[-1] = ("", ps, e)
        else:
            merged.append((lab, s, e))

    return merged


def get_tier(tg, name):

    for tier in tg.tiers:
        if tier.name == name:
            return tier

    raise ValueError(f"Tier {name} not found")


def extract_intervals(tier, chunk_start, chunk_end):

    result = []

    for iv in tier.intervals:

        start = iv.minTime
        end = iv.maxTime
        label = iv.mark.strip()

        if end <= chunk_start:
            continue

        if start >= chunk_end:
            continue

        ov_start = max(start, chunk_start)
        ov_end = min(end, chunk_end)

        rel_start = ov_start - chunk_start
        rel_end = ov_end - chunk_start

        result.append(
            (
                label,
                float(rel_start),
                float(rel_end),
            )
        )

    return merge_empty_intervals(result)


def build_language(lang_dir):

    lang = lang_dir.name

    print(f"Processing {lang}")

    csv_path = lang_dir / "chunks.csv"

    audio_dir = lang_dir / "audio"
    tg_dir = lang_dir / "alignments"

    chunks = pd.read_csv(csv_path)

    examples = []

    cached_audio = {}
    cached_tg = {}

    for _, row in chunks.iterrows():

        filename = row["filename"]

        chunk_start = float(row["start"])
        chunk_end = float(row["end"])

        text = normalize_text(row["text"])

        if not valid_chunk(text):
            continue

        wav_path = audio_dir / filename

        tg_path = tg_dir / (
            Path(filename).stem + ".TextGrid"
        )

        if filename not in cached_audio:

            audio, sr = sf.read(wav_path)

            if audio.ndim > 1:
                audio = audio.mean(axis=1)

            cached_audio[filename] = (audio, sr)

        audio, sr = cached_audio[filename]

        if tg_path not in cached_tg:

            tg = TextGrid()
            tg.read(str(tg_path))

            cached_tg[tg_path] = tg

        tg = cached_tg[tg_path]

        start_sample = int(chunk_start * sr)
        end_sample = int(chunk_end * sr)

        chunk_audio = audio[start_sample:end_sample]

        if sr != TARGET_SR:

            chunk_audio = librosa.resample(
                chunk_audio,
                orig_sr=sr,
                target_sr=TARGET_SR,
            )

        chunk_audio = chunk_audio.astype(np.float32)

        wd_tier = get_tier(tg, "wd")
        ph_tier = get_tier(tg, "ph")

        words = []

        for label, s, e in extract_intervals(
            wd_tier,
            chunk_start,
            chunk_end,
        ):

            label = label.strip()

            if label == "":
                continue

            words.append(
                {
                    "text": label,
                    "start": s,
                    "end": e,
                }
            )

        phones = []

        for label, s, e in extract_intervals(
            ph_tier,
            chunk_start,
            chunk_end,
        ):

            label = label.strip()

            if label == "":
                continue

            phones.append(
                {
                    "phone": label,
                    "start": s,
                    "end": e,
                }
            )

        if len(words) <= 1 or len(phones) < len(words):
            continue

        examples.append(
            {
                "audio": {
                    "array": chunk_audio,
                    "sampling_rate": TARGET_SR,
                },
                "transcription": text,
                "words": words,
                "phones": phones,
                "language": lang,
            }
        )

    return examples

In [43]:
all_examples = build_language(
    Path("processed/stan1290")
)

Processing stan1290


In [44]:
from IPython.display import Audio, display
import numpy as np


def play_segment(audio_array, sampling_rate, start=None, end=None):
    """
    Play a segment of an audio array.

    Parameters
    ----------
    audio_array : np.ndarray
        1D float audio array
    sampling_rate : int
        Sampling rate
    start : float or None
        Start time in seconds
    end : float or None
        End time in seconds
    """

    if start is None:
        start = 0.0

    if end is None:
        end = len(audio_array) / sampling_rate

    start_idx = max(0, int(start * sampling_rate))
    end_idx = min(len(audio_array), int(end * sampling_rate))

    segment = audio_array[start_idx:end_idx]

    display(Audio(segment, rate=sampling_rate))

In [45]:
ex = all_examples[0]

play_segment(
    ex["audio"]["array"],
    ex["audio"]["sampling_rate"]
)

In [46]:
import random
from pathlib import Path

import numpy as np
import pandas as pd
from datasets import Dataset


RANDOM_SEED = 42

TRAIN_RATIO = 0.7
VAL_RATIO = 0.1
TEST_RATIO = 0.2

assert abs(TRAIN_RATIO + VAL_RATIO + TEST_RATIO - 1.0) < 1e-8


def split_examples(examples,
                   train_ratio=TRAIN_RATIO,
                   val_ratio=VAL_RATIO,
                   test_ratio=TEST_RATIO,
                   seed=RANDOM_SEED):

    rng = random.Random(seed)

    indices = list(range(len(examples)))
    rng.shuffle(indices)

    n = len(indices)

    n_train = int(round(n * train_ratio))
    n_val = int(round(n * val_ratio))

    if n_train >= n:
        n_train = max(1, n - 2)

    if n_train + n_val >= n:
        n_val = max(1, n - n_train - 1)

    train_idx = indices[:n_train]
    val_idx = indices[n_train:n_train + n_val]
    test_idx = indices[n_train + n_val:]

    if len(test_idx) == 0 and n >= 3:
        test_idx = [train_idx.pop()]

    return (
        [examples[i] for i in train_idx],
        [examples[i] for i in val_idx],
        [examples[i] for i in test_idx],
    )


def examples_to_dataframe(examples):

    rows = []

    for ex in examples:
        rows.append(
            {
                "audio": {
                    "array": ex["audio"]["array"].tolist(),
                    "sampling_rate": ex["audio"]["sampling_rate"],
                },
                "transcription": ex["transcription"],
                "words": ex["words"],
                "phones": ex["phones"],
                "language": ex["language"],
            }
        )

    return pd.DataFrame(rows)


def write_split(examples, output_file):

    df = examples_to_dataframe(examples)

    table = Dataset.from_pandas(
        df,
        preserve_index=False,
    )

    table.to_parquet(str(output_file))


def process_language_directory(lang_dir, output_root):

    lang = lang_dir.name

    print(f"\n=== {lang} ===")

    examples = build_language(lang_dir)

    print(f"Valid examples: {len(examples)}")

    if len(examples) == 0:
        print("Skipping empty language")
        return

    train_examples, val_examples, test_examples = split_examples(
        examples
    )

    lang_out = output_root / lang
    lang_out.mkdir(parents=True, exist_ok=True)

    write_split(
        train_examples,
        lang_out / "train.parquet",
    )

    write_split(
        val_examples,
        lang_out / "validation.parquet",
    )

    write_split(
        test_examples,
        lang_out / "test.parquet",
    )

    print(
        f"train={len(train_examples)} "
        f"val={len(val_examples)} "
        f"test={len(test_examples)}"
    )


def write_dataset_script(output_root):

    script = r'''
import datasets
from pathlib import Path


_LANGS = sorted(
    [
        p.name
        for p in Path(__file__).parent.iterdir()
        if p.is_dir()
    ]
)


class DoReCoConfig(datasets.BuilderConfig):
    pass


class DoReCo(datasets.GeneratorBasedBuilder):

    BUILDER_CONFIGS = [
        DoReCoConfig(
            name="all",
            version=datasets.Version("1.0.0"),
        )
    ] + [
        DoReCoConfig(
            name=lang,
            version=datasets.Version("1.0.0"),
        )
        for lang in _LANGS
    ]

    DEFAULT_CONFIG_NAME = "all"

    def _info(self):

        return datasets.DatasetInfo(
            features=datasets.Features(
                {
                    "audio": datasets.Audio(
                        sampling_rate=16000
                    ),
                    "transcription": datasets.Value(
                        "string"
                    ),
                    "words": datasets.Sequence(
                        {
                            "text": datasets.Value(
                                "string"
                            ),
                            "start": datasets.Value(
                                "float32"
                            ),
                            "end": datasets.Value(
                                "float32"
                            ),
                        }
                    ),
                    "phones": datasets.Sequence(
                        {
                            "phone": datasets.Value(
                                "string"
                            ),
                            "start": datasets.Value(
                                "float32"
                            ),
                            "end": datasets.Value(
                                "float32"
                            ),
                        }
                    ),
                    "language": datasets.Value(
                        "string"
                    ),
                }
            )
        )

    def _split_generators(self, dl_manager):

        root = Path(__file__).parent

        if self.config.name == "all":
            langs = _LANGS
        else:
            langs = [self.config.name]

        return [
            datasets.SplitGenerator(
                name=datasets.Split.TRAIN,
                gen_kwargs={
                    "split_name": "train",
                    "langs": langs,
                    "root": root,
                },
            ),
            datasets.SplitGenerator(
                name=datasets.Split.VALIDATION,
                gen_kwargs={
                    "split_name": "validation",
                    "langs": langs,
                    "root": root,
                },
            ),
            datasets.SplitGenerator(
                name=datasets.Split.TEST,
                gen_kwargs={
                    "split_name": "test",
                    "langs": langs,
                    "root": root,
                },
            ),
        ]

    def _generate_examples(
        self,
        split_name,
        langs,
        root,
    ):

        idx = 0

        for lang in langs:

            parquet_file = (
                root
                / lang
                / f"{split_name}.parquet"
            )

            if not parquet_file.exists():
                continue

            ds = datasets.Dataset.from_parquet(
                str(parquet_file)
            )

            for row in ds:

                yield idx, row
                idx += 1
'''

    with open(
        output_root / "doreco.py",
        "w",
        encoding="utf-8",
    ) as f:
        f.write(script)


def build_doreco_dataset(
    processed_root="processed",
    output_root="doreco_dataset",
):

    processed_root = Path(processed_root)
    output_root = Path(output_root)

    output_root.mkdir(
        parents=True,
        exist_ok=True,
    )

    langs = sorted(
        [
            p
            for p in processed_root.iterdir()
            if p.is_dir()
        ]
    )

    print(f"Languages found: {len(langs)}")

    for lang_dir in langs:
        process_language_directory(
            lang_dir,
            output_root,
        )

    write_dataset_script(output_root)

    print("\nDataset written to:")
    print(output_root.resolve())

    print("\nExample usage:")
    print(
        'load_dataset("doreco_dataset", lang_dir="stan1290")'
    )
    print(
        'load_dataset("doreco_dataset")'
    )


if __name__ == "__main__":
    build_doreco_dataset()

Languages found: 45

=== anal1239 ===
Processing anal1239
Valid examples: 2268


Creating parquet from Arrow format:   0%|          | 0/2 [00:00<?, ?ba/s]

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

train=1588 val=227 test=453

=== apah1238 ===
Processing apah1238
Valid examples: 1510


...

train=465 val=66 test=133

=== yuca1254 ===
Processing yuca1254
Valid examples: 1125


Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

train=788 val=112 test=225

Dataset written to:
/media/mahesh-akavarapu/EXTRA/MFA/doreco/doreco_dataset

Example usage:
load_dataset("doreco_dataset", "stan1290")
load_dataset("doreco_dataset", "all")


In [47]:
from datasets import load_dataset

In [57]:
dataset = load_dataset("doreco_dataset", data_dir="anal1239", streaming=False)

Generating train split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]

In [58]:
dataset['test'][0]

{'audio': {'array': [0.003451626282185316,
   0.006318212952464819,
   0.006572263315320015,
   0.006502252072095871,
   0.0044944193214178085,
   0.0045195636339485645,
   0.0030648154206573963,
   0.0026810537092387676,
   0.0025465376675128937,
   0.0018172094132751226,
   0.001330515369772911,
   0.0023352261632680893,
   0.0014100169064477086,
   0.0005854517221450806,
   7.281772559508681e-05,
   -0.0014189549256116152,
   -0.0027105649933218956,
   -0.005129425320774317,
   -0.004186294972896576,
   -0.005408963188529015,
   -0.0056657129898667336,
   -0.003871225519105792,
   -0.005546987988054752,
   -0.008439743891358376,
   -0.008111798204481602,
   -0.007439759559929371,
   -0.007575988303869963,
   -0.008695683442056179,
   -0.009013315662741661,
   -0.00870833732187748,
   -0.009713317267596722,
   -0.009262772276997566,
   -0.009573300369083881,
   -0.009974710643291473,
   -0.009489239193499088,
   -0.00822446495294571,
   -0.007880071178078651,
   -0.008062014356255531

In [60]:
play_segment(dataset['test'][0]['audio']['array'], dataset['test'][0]['audio']['sampling_rate'], 1.52, 1.68)